In [2]:
import pandas as pd
import sklearn

In [3]:
df= pd.read_csv("retail_cleaned2.csv")
df

C:\Users\sivanand\AppData\Local\Temp\ipykernel_21076\3041861155.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.read_csv("retail_cleaned2.csv")


,Invoice,StockCode,Description,Quantity,Revenue,Country,Month,Year
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,83.40,United Kingdom,12,2009
1,489434,79323P,PINK CHERRY LIGHTS,12,81.00,United Kingdom,12,2009
2,489434,79323W,WHITE CHERRY LIGHTS,12,81.00,United Kingdom,12,2009
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,100.80,United Kingdom,12,2009
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,30.00,United Kingdom,12,2009
...,...,...,...,...,...,...,...,...
1041665,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12.60,France,12,2011
1041666,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,16.60,France,12,2011
1041667,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,16.60,France,12,2011
1041668,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,14.85,France,12,2011


In [4]:
df = df.drop(columns=['Invoice'])

In [5]:
from sklearn.preprocessing import LabelEncoder

top_countries = df['Country'].value_counts()
top_countries = top_countries[top_countries >= 500].index

df['Country_Grouped'] = df['Country'].apply(
    lambda x: x if x in top_countries else 'Other'
)

le = LabelEncoder()
df['Country_Encoded'] = le.fit_transform(df['Country_Grouped'])

In [6]:
df

,StockCode,Description,Quantity,Revenue,Country,Month,Year,Country_Grouped,Country_Encoded
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,83.40,United Kingdom,12,2009,United Kingdom,20
1,79323P,PINK CHERRY LIGHTS,12,81.00,United Kingdom,12,2009,United Kingdom,20
2,79323W,WHITE CHERRY LIGHTS,12,81.00,United Kingdom,12,2009,United Kingdom,20
3,22041,"RECORD FRAME 7"" SINGLE SIZE",48,100.80,United Kingdom,12,2009,United Kingdom,20
4,21232,STRAWBERRY CERAMIC TRINKET BOX,24,30.00,United Kingdom,12,2009,United Kingdom,20
...,...,...,...,...,...,...,...,...,...
1041665,22899,CHILDREN'S APRON DOLLY GIRL,6,12.60,France,12,2011,France,8
1041666,23254,CHILDRENS CUTLERY DOLLY GIRL,4,16.60,France,12,2011,France,8
1041667,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,16.60,France,12,2011,France,8
1041668,22138,BAKING SET 9 PIECE RETROSPOT,3,14.85,France,12,2011,France,8


In [7]:
df['StockCode'] = df['StockCode'].str.upper().str.strip()

non_products = ['POST', 'DOT', 'M', 'BANK CHARGES', 'PADS', 'AMAZONFEE']
df = df[~df['StockCode'].isin(non_products)]
df = df[df['StockCode'].str.match(r'^\d{5}[a-zA-Z]{0,2}$', na=False)]

In [8]:
from sklearn.preprocessing import LabelEncoder

#Create product lookup BEFORE encoding
product_lookup = df[['StockCode', 'Description']].drop_duplicates().reset_index(drop=True)
product_lookup.to_csv('product_lookup.csv', index=False)
print(f"Product lookup saved — {len(product_lookup)} unique products")

#Encode StockCode
le_stock = LabelEncoder()
df['StockCode_Encoded'] = le_stock.fit_transform(df['StockCode'])

#Add encoded number to lookup table too
product_lookup['StockCode_Encoded'] = le_stock.transform(product_lookup['StockCode'])
product_lookup = product_lookup.sort_values('StockCode_Encoded').reset_index(drop=True)

print(f"\nProduct lookup with encoding:")
print(product_lookup.head(10))

Product lookup saved — 5391 unique products

Product lookup with encoding:
  StockCode                  Description  StockCode_Encoded
0     10002  INFLATABLE POLITICAL GLOBE                   0
1    10002R        ROBOT PENCIL SHARPNER                  1
2     10080     GROOVY CACTUS INFLATABLE                  2
3     10109         BENDY COLOUR PENCILS                  3
4     10120                 DOGGY RUBBER                  4
5    10123C        HEARTS WRAPPING TAPE                   5
6    10123G      ARMY CAMO WRAPPING TAPE                  6
7    10124A  SPOTS ON RED BOOKCOVER TAPE                  7
8    10124G     ARMY CAMO BOOKCOVER TAPE                  8
9     10125      MINI FUNKY DESIGN TAPES                  9


In [9]:
from sklearn.preprocessing import LabelEncoder

#  Clean Description text
df['Description'] = df['Description'].str.upper().str.strip()

#For each StockCode keep only the most common description
# This fixes the duplicate wording problem
df['Description'] = df.groupby('StockCode')['Description']\
                       .transform(lambda x: x.mode()[0])

print(f"Unique descriptions after cleaning: {df['Description'].nunique()}")

# Encode Description
le_desc = LabelEncoder()
df['Description_Encoded'] = le_desc.fit_transform(df['Description'])

#Create lookup table
product_lookup = df[['Description_Encoded', 'StockCode', 'Description']]\
                   .drop_duplicates()\
                   .sort_values('Description_Encoded')\
                   .reset_index(drop=True)

product_lookup.to_csv('product_lookup.csv', index=False)

print(f"\nSample lookup table:")
print(product_lookup.head(10))

Unique descriptions after cleaning: 4671

Sample lookup table:
   Description_Encoded StockCode                        Description
0                    0     21120              *BOOMBOX IPOD CLASSIC
1                    1     20739           *USB OFFICE GLITTER LAMP
2                    2     20954            *USB OFFICE MIRROR BALL
3                    3     22418             10 COLOUR SPACEBOY PEN
4                    4     35962  12 ASS ZINC CHRISTMAS DECORATIONS
5                    5     22436         12 COLOURED PARTY BALLOONS
6                    6     21448          12 DAISY PEGS IN WOOD BOX
7                    7     22282          12 EGG HOUSE PAINTED WOOD
8                    8     23442       12 HANGING EGGS HAND PAINTED
9                    9     21447   12 IVORY ROSE PEG PLACE SETTINGS


In [10]:
import pickle

# Save both encoders
with open('le_country.pkl', 'wb') as f:
    pickle.dump(le, f)

with open('le_desc.pkl', 'wb') as f:
    pickle.dump(le_desc, f)

print("Encoders saved!")

Encoders saved!


In [11]:
df

,StockCode,Description,Quantity,Revenue,Country,Month,Year,Country_Grouped,Country_Encoded,StockCode_Encoded,Description_Encoded
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,83.40,United Kingdom,12,2009,United Kingdom,20,4080,24
1,79323P,PINK CHERRY LIGHTS,12,81.00,United Kingdom,12,2009,United Kingdom,20,3396,2869
2,79323W,WHITE CHERRY LIGHTS,12,81.00,United Kingdom,12,2009,United Kingdom,20,3398,4436
3,22041,"RECORD FRAME 7"" SINGLE SIZE",48,100.80,United Kingdom,12,2009,United Kingdom,20,1275,3174
4,21232,STRAWBERRY CERAMIC TRINKET BOX,24,30.00,United Kingdom,12,2009,United Kingdom,20,626,4059
...,...,...,...,...,...,...,...,...,...,...,...
1041664,22613,PACK OF 20 SPACEBOY NAPKINS,12,10.20,France,12,2011,France,8,1812,2669
1041665,22899,CHILDREN'S APRON DOLLY GIRL,6,12.60,France,12,2011,France,8,2092,850
1041666,23254,CHILDRENS CUTLERY DOLLY GIRL,4,16.60,France,12,2011,France,8,2432,856
1041667,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,16.60,France,12,2011,France,8,2433,855


In [12]:
df.to_csv('retail_processes.csv', index=False)

In [16]:
df=df.drop('StockCode_Encoded', axis=1)